<a href="https://colab.research.google.com/github/MarlzRana/machine-learning/blob/main/gta_pretrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GTA Pretrain
In this notebook we will pre-training our model on the GTA segmentation dataset

## Imports

External imports

In [ ]:
import albumentations as A

import torch

from torch.utils.data import DataLoader

from torch.optim.lr_scheduler import ReduceLROnPlateau

from matplotlib import pyplot as plt

## Internal imports

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/dataset.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/mod_unet.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/loss.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/epochs.ipynb"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/logger.ipynb"

## Constants

In [ ]:
DEVICE=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

DS_FILE_PTH = "./gta.h5"

PROJECT_BASE_PTH="/content/drive/My Drive/3yp/"

PROJECT_CHECKPOINTS_PTH=PROJECT_BASE_PTH+"checkpoints/"
CHECKPOINT_FEXT=".pth"

PROJECT_LOGS_PTH=PROJECT_BASE_PTH+"logs/"
LOG_FEXT=".csv"

CHECKPOINTS_PTH=PROJECT_CHECKPOINTS_PTH+"gta/"
LOGS_PTH=PROJECT_LOGS_PTH+"gta/"

RESNET18_UNET_FNAME="resnet18_unet"
RESNET34_UNET_FNAME="resnet34_unet"
RESNET50_UNET_FNAME="resnet50_unet"
RESNET101_UNET_FNAME="resnet101_unet"
RESNET152_UNET_FNAME="resnet152_unet"

PRETRAIN_RESNET18_UNET_FNAME="pretrain_resnet18_unet"
PRETRAIN_RESNET34_UNET_FNAME="pretrain_resnet34_unet"
PRETRAIN_RESNET50_UNET_FNAME="pretrain_resnet50_unet"
PRETRAIN_RESNET101_UNET_FNAME="pretrain_resnet101_unet"
PRETRAIN_RESNET152_UNET_FNAME="pretrain_resnet152_unet"

LBL_MAP = {
    "road": 0,
    "sidewalk": 1,
    "building": 2,
    "wall": 3,
    "fence": 4,
    "pole": 5,
    "traffic light": 6,
    "traffic sign": 7,
    "vegetation": 8,
    "terrain": 9,
    "sky": 10,
    "person": 11,
    "rider": 12,
    "car": 13,
    "truck": 14,
    "bus": 15,
    "train": 16,
    "motorcycle": 17,
    "bicycle": 18
}

NUM_CLASSES=len(LBL_MAP)

TRAIN_BATCH_SIZE=32
VAL_BATCH_SIZE=32
TEST_BATCH_SIZE=32

TRAIN_DL_WORKERS=2
VAL_DL_WORKERS=2

LR=1e-4

LR_SCH_PATIENCE=3

NUM_EPOCHS=30

## Copy dataset to local storage

In [ ]:
!cp "/content/drive/My Drive/3yp/datasets/gta.h5" .

## Dataset Transformations

Numpy to Torch Tensor transformation

In [ ]:
def to_tensor(x, cols, rows):
  return x.transpose(2, 0, 1)

Image transformation

In [ ]:
img_tr = A.Compose(
    [
      A.Normalize(mean=0.0, std=1.0), # 0/1 Normalization
      A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=1), # ImageNet Normalization,
      A.Lambda(image=to_tensor)
    ]
)

Mask transformation

In [ ]:
msk_tr = A.Compose([
  A.Lambda(image=to_tensor, mask=to_tensor)
])

## Datasets

Training dataset

In [ ]:
train_ds = HDF5MultipleSegmentationDatasetsSeperateMsks(
  ds_map={
    "gta": {
      "h5_file_pth": DS_FILE_PTH,
      "sub_ds_name": "train"
      },
    },
  lbl_map=LBL_MAP,
  img_post_tr=img_tr,
  msk_post_tr=msk_tr
)

Validation dataset

In [ ]:
val_ds = HDF5MultipleSegmentationDatasetsSeperateMsks(
  ds_map={
    "gta": {
      "h5_file_pth": DS_FILE_PTH,
      "sub_ds_name": "val"
      },
    },
  lbl_map=LBL_MAP,
  img_post_tr=img_tr,
  msk_post_tr=msk_tr
)

## Dataloaders

In [ ]:
train_dl = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=TRAIN_DL_WORKERS)

In [ ]:
val_dl = DataLoader(val_ds, batch_size=VAL_BATCH_SIZE, shuffle=True, num_workers=VAL_DL_WORKERS)

## Loss

In [ ]:
loss_fn = DiceLoss()

## Train

Pretrained ResNet18_UNet model

In [ ]:
model_pretrain_resnet18_unet = torch.load(CHECKPOINTS_PTH + PRETRAIN_RESNET18_UNET_FNAME + CHECKPOINT_FEXT).to(DEVICE)

In [ ]:
optim_pretrain_resnet18_unet = torch.optim.Adam(
    model_pretrain_resnet18_unet.parameters(),
    lr=LR
)

In [ ]:
lr_sch_pretrain_resnet18_unet = ReduceLROnPlateau(optim_pretrain_resnet18_unet, mode="min", patience=LR_SCH_PATIENCE, factor=1e-1, verbose=True)

In [ ]:
logger_pretrain_resnet18_unet = EpochLogger(LOGS_PTH + PRETRAIN_RESNET18_UNET_FNAME + LOG_FEXT)

In [ ]:
num_ran_epochs = logger_pretrain_resnet18_unet.epoch_num

In [ ]:
train_epoch_pretrain_resnet18_unet = TrainEpoch(
    dataloader=train_dl,
    model=model_pretrain_resnet18_unet,
    optimizer=optim_pretrain_resnet18_unet,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE,
    num_ran_epochs=num_ran_epochs
)

In [ ]:
val_epoch_pretrain_resnet18_unet = EvalEpoch(
    dataloader=val_dl,
    model=model_pretrain_resnet18_unet,
    loss_fn=loss_fn,
    metrics=[],
    device=DEVICE,
    num_ran_epochs=num_ran_epochs
)

In [ ]:
min_val_loss = None

if len(logger_pretrain_resnet18_unet.df_log) > 0:
  min_val_loss = min(logger_pretrain_resnet18_unet.df_log["Val Loss"])
  for val_loss in logger_pretrain_resnet18_unet.df_log["Val Loss"]:
    lr_sch_pretrain_resnet18_unet.step(val_loss)
else:
  print("Calculating the initial validation loss")
  min_val_loss = val_epoch_pretrain_resnet18_unet.run()
  print(f"Initial validation loss: {min_val_loss}\n")

In [ ]:
for i in range(num_ran_epochs, NUM_EPOCHS):
  print(f"Epoch {i+1}:")

  train_loss = train_epoch_pretrain_resnet18_unet.run()
  val_loss = val_epoch_pretrain_resnet18_unet.run()

  model_saved = False

  if (val_loss < min_val_loss):
    print(f"Saved the {i+1}th model which had a val_loss of {val_loss}")
    torch.save(model_pretrain_resnet18_unet, CHECKPOINTS_PTH + PRETRAIN_RESNET18_UNET_FNAME + CHECKPOINT_FEXT)
    min_val_loss = val_loss
    model_saved=True

  logger_pretrain_resnet18_unet.add_row(
      train_loss=train_loss,
      val_loss=val_loss,
      lr=optim_pretrain_resnet18_unet.param_groups[0]['lr'],
      model_saved=model_saved,
      train_batch_size=TRAIN_BATCH_SIZE,
      val_batch_size=VAL_BATCH_SIZE
  )

  lr_sch_pretrain_resnet18_unet.step(val_loss)
  num_ran_epochs += 1

  print("")

Epoch 17:


100%|██████████| 3993/3993 [04:44<00:00, 14.02it/s, eval_loss=0.104]



Epoch 18:


100%|██████████| 3993/3993 [04:39<00:00, 14.28it/s, eval_loss=0.102]



Epoch 19:


...

100%|██████████| 3993/3993 [04:43<00:00, 14.06it/s, eval_loss=0.0943]


Saved the 22th model which had a val_loss of 0.09425366675968265

Epoch 23:


100%|██████████| 3993/3993 [04:39<00:00, 14.29it/s, eval_loss=0.101]



Epoch 24:


 20%|█▉        | 3136/15978 [03:45<13:13, 16.19it/s, train_loss=0.0862]

In [ ]:
LOGS_PTH + PRETRAIN_RESNET18_UNET_FNAME + LOG_FEXT